### Purpose

**글로벌 DB 봇**과 웹 승인 배포 (docs/gdb-request-registration-queue.md 4차 개정 — 4.8, 10장 G7~G11).

| 대상 | 내용 |
|---|---|
| SSM `/dsglobaldb/bot/token` (SecureString) | 봇 토큰 — `AWS/Lambda/.env`의 `GDB_BOT_TOKEN`에서 올린다. 코드·출력에는 남기지 않음 |
| DynamoDB `dsglobaldb-bot` | 보낸 카드(`card#종류#코드`), 사유 대기(`reason#…`, TTL 1일) |
| Lambda `dsglobaldbBot` (새로) | 텔레그램 webhook `POST /tgbot` + create·결정 알림(비동기) + 아침 요약 |
| Lambda `dsglobaldbCreate` (갱신) | 웹 create 승인 없음, 봇 알림, `GET/POST /regreview`(Cognito), `/internal/*`(봇 전용) |
| API Gateway | `/tgbot`(키 없음, secret_token 헤더로 확인), `/regreview`(Cognito 권한 부여자) |
| EventBridge `dsglobaldb-bot-morning` | 매일 08:00 KST → 승인 대기 요약 |
| IAM `DSmapLambda` | 인라인 `DSGlobalDBBot`: SSM 읽기, 두 Lambda 서로 호출, 봇 테이블 |

**`.env` (AWS/Lambda/.env, git 제외)**
- `GDB_BOT_TOKEN` — BotFather 토큰
- `GDB_OWNER_EMAILS` — 웹 승인자(Cognito 로그인 이메일), 쉼표
- `GDB_OWNER_TG` — 봇 승인자 텔레그램 번호(`번호` 또는 `번호:이름`), 쉼표. **비어 있으면** 배포 뒤 봇에게 `/start` → 번호를 받아 넣고 **6번 셀만** 다시 실행
- `GDB_BOT_WEBHOOK_SECRET` — 처음 실행 때 노트북이 만들어 넣는다

- **마지막 셀 전의 스테이지 배포는 API 전체를 다시 배포한다**

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import json
import secrets
import time
import urllib.error
import urllib.request
from pathlib import Path

import boto3

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
from deploy_helpers import (deploy_lambda, ensure_resource_path, get_request_template, setup_lambda_method,
                            setup_options, grant_invoke, ensure_cognito_authorizer)

Settings + `.env` 읽기 (값은 출력하지 않음)

In [ ]:
ROLE_NAME = 'DSmapLambda'
ROLE_ARN = 'arn:aws:iam::199525800117:role/DSmapLambda'
ACCOUNT_ID = ROLE_ARN.split(':')[4]
API_ID = 'jyipsj28s9'
STAGE = 'dev'
TEMPLATE_FROM = '/dschemical'
BASE_URL = f'https://{API_ID}.execute-api.us-east-1.amazonaws.com/{STAGE}'

CORE_FUNCTION = 'dsglobaldbCreate'
CORE_SOURCE = './dsglobaldbCreate/lambda_function.py'
BOT_FUNCTION = 'dsglobaldbBot'
BOT_SOURCE = './dsglobaldbBot/lambda_function.py'
BOT_TABLE = 'dsglobaldb-bot'
TOKEN_PARAM = '/dsglobaldb/bot/token'
POLICY_NAME = 'DSGlobalDBBot'
USER_POOL_ARN = f'arn:aws:cognito-idp:us-east-1:{ACCOUNT_ID}:userpool/us-east-1_W0IaBru9e'   # src/main.jsx 의 사용자 풀
AUTHORIZER_NAME = 'dsglobaldb-cognito'
MORNING_RULE = 'dsglobaldb-bot-morning'
MORNING_CRON = 'cron(0 23 * * ? *)'          # 23:00 UTC = 08:00 KST
AMPLIFY_APP = 'dsglobaldb_manager_web'

ENV_PATH = Path('.env')

def read_env():
    values = {}
    for line in ENV_PATH.read_text(encoding='utf-8').splitlines():
        if line.strip() and not line.lstrip().startswith('#') and '=' in line:
            k, _, v = line.partition('=')
            values[k.strip()] = v.strip()
    return values

def write_env_value(key, value):
    lines = [l for l in ENV_PATH.read_text(encoding='utf-8').splitlines() if not l.startswith(f'{key}=')]
    ENV_PATH.write_text('\n'.join(lines + [f'{key}={value}']) + '\n', encoding='utf-8')

env = read_env()
assert env.get('GDB_BOT_TOKEN'), '.env 의 GDB_BOT_TOKEN 이 비어 있습니다'
assert env.get('GDB_OWNER_EMAILS'), '.env 의 GDB_OWNER_EMAILS 가 비어 있습니다'
if not env.get('GDB_BOT_WEBHOOK_SECRET'):
    write_env_value('GDB_BOT_WEBHOOK_SECRET', secrets.token_urlsafe(32))
    env = read_env()
    print('webhook secret 을 새로 만들어 .env 에 넣음')
TOKEN = env['GDB_BOT_TOKEN']

def tg(method, **params):
    req = urllib.request.Request(f'https://api.telegram.org/bot{TOKEN}/{method}', method='POST',
                                 data=json.dumps(params).encode(), headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req) as r:
        return json.load(r)['result']

me = tg('getMe')
print('봇:', '@' + me['username'], me['first_name'])
print('웹 승인자:', len(env['GDB_OWNER_EMAILS'].split(',')), '명 · 봇 승인자:',
      len([x for x in env.get('GDB_OWNER_TG', '').split(',') if x.strip()]), '명')

amplify_app = next(a for a in boto3.client('amplify').list_apps()['apps'] if a['name'] == AMPLIFY_APP)
WEB_URL = f"https://main.{amplify_app['defaultDomain']}/dsdb/regrequests"
print('웹:', WEB_URL)

1. SSM 토큰 · DynamoDB 테이블(TTL) · IAM

In [ ]:
boto3.client('ssm').put_parameter(Name=TOKEN_PARAM, Value=TOKEN, Type='SecureString', Overwrite=True,
                                  Description='글로벌 DB 봇 토큰 (dsglobaldbBot)')
print('SSM:', TOKEN_PARAM)

ddb = boto3.client('dynamodb')
if BOT_TABLE not in ddb.list_tables()['TableNames']:
    ddb.create_table(TableName=BOT_TABLE, BillingMode='PAY_PER_REQUEST',
                     KeySchema=[{'AttributeName': 'pk', 'KeyType': 'HASH'}, {'AttributeName': 'sk', 'KeyType': 'RANGE'}],
                     AttributeDefinitions=[{'AttributeName': 'pk', 'AttributeType': 'S'}, {'AttributeName': 'sk', 'AttributeType': 'S'}])
    ddb.get_waiter('table_exists').wait(TableName=BOT_TABLE)
    print('테이블 생성:', BOT_TABLE)
else:
    print('테이블 있음:', BOT_TABLE)
if ddb.describe_time_to_live(TableName=BOT_TABLE)['TimeToLiveDescription']['TimeToLiveStatus'] in ('DISABLED',):
    ddb.update_time_to_live(TableName=BOT_TABLE, TimeToLiveSpecification={'Enabled': True, 'AttributeName': 'ttl'})
    print('TTL 켬: ttl')

boto3.client('iam').put_role_policy(RoleName=ROLE_NAME, PolicyName=POLICY_NAME, PolicyDocument=json.dumps({
    'Version': '2012-10-17',
    'Statement': [
        {'Effect': 'Allow', 'Action': ['ssm:GetParameter'],
         'Resource': f'arn:aws:ssm:us-east-1:{ACCOUNT_ID}:parameter{TOKEN_PARAM}'},
        {'Effect': 'Allow', 'Action': ['lambda:InvokeFunction'],
         'Resource': [f'arn:aws:lambda:us-east-1:{ACCOUNT_ID}:function:{CORE_FUNCTION}',
                      f'arn:aws:lambda:us-east-1:{ACCOUNT_ID}:function:{BOT_FUNCTION}']},
        {'Effect': 'Allow', 'Action': ['dynamodb:GetItem', 'dynamodb:PutItem', 'dynamodb:DeleteItem', 'dynamodb:Query'],
         'Resource': f'arn:aws:dynamodb:us-east-1:{ACCOUNT_ID}:table/{BOT_TABLE}'},
    ],
}))
print('인라인 정책:', ROLE_NAME, POLICY_NAME)
time.sleep(10)    # IAM 반영 대기

2. Lambda — 봇(새로) · core(갱신) + 환경 변수

In [ ]:
lam = boto3.client('lambda')

def set_env(function, values):
    current = lam.get_function_configuration(FunctionName=function).get('Environment', {}).get('Variables', {})
    lam.update_function_configuration(FunctionName=function, Environment={'Variables': {**current, **values}})
    lam.get_waiter('function_updated_v2').wait(FunctionName=function)

bot_arn = deploy_lambda(BOT_FUNCTION, BOT_SOURCE, ROLE_ARN, description='글로벌 DB 봇 — 등록 승인 카드 (텔레그램 webhook)',
                        timeout=29, memory=256)
set_env(BOT_FUNCTION, {'TOKEN_PARAM': TOKEN_PARAM, 'WEBHOOK_SECRET': env['GDB_BOT_WEBHOOK_SECRET'],
                       'GDB_OWNER_TG': env.get('GDB_OWNER_TG', ''), 'CORE_FUNCTION': CORE_FUNCTION,
                       'BOT_TABLE': BOT_TABLE, 'WEB_URL': WEB_URL})
core_arn = deploy_lambda(CORE_FUNCTION, CORE_SOURCE, ROLE_ARN,
                         description='글로벌 DB 안전한 쓰기 (create · update/alias/history · 등록 요청 review)', timeout=29, memory=256)
set_env(CORE_FUNCTION, {'GDB_BOT_FUNCTION': BOT_FUNCTION, 'GDB_OWNER_EMAILS': env['GDB_OWNER_EMAILS']})
print('환경 변수 설정 끝')

3. API Gateway — `/tgbot`(봇) · `/regreview`(Cognito, core)

In [ ]:
apigw = boto3.client('apigateway')
passthrough = get_request_template(API_ID, TEMPLATE_FROM)
# /regreview: 권한 부여자가 확인한 로그인 사람(claims)을 Lambda 에 넘긴다
REVIEW_TEMPLATE = '''{
  "body-json" : $input.json('$'),
  "params" : { "header" : {} },
  "context" : {
    "http-method" : "$context.httpMethod",
    "resource-path" : "$context.resourcePath",
    "claims-email" : "$util.escapeJavaScript($context.authorizer.claims.email)",
    "claims-name" : "$util.escapeJavaScript($context.authorizer.claims.name)",
    "claims-sub" : "$util.escapeJavaScript($context.authorizer.claims.sub)"
  }
}'''
authorizer_id = ensure_cognito_authorizer(API_ID, AUTHORIZER_NAME, USER_POOL_ARN)

rid = ensure_resource_path(API_ID, 'tgbot')
setup_lambda_method(API_ID, rid, 'POST', bot_arn, passthrough)
grant_invoke(BOT_FUNCTION, API_ID, ACCOUNT_ID, 'tgbot')

rid = ensure_resource_path(API_ID, 'regreview')
for method in ('GET', 'POST'):
    setup_lambda_method(API_ID, rid, method, core_arn, REVIEW_TEMPLATE, authorizer_id=authorizer_id)
setup_options(API_ID, rid, ['GET', 'POST'])
grant_invoke(CORE_FUNCTION, API_ID, ACCOUNT_ID, 'regreview')

4. 아침 요약 — EventBridge 매일 08:00 KST

In [ ]:
events = boto3.client('events')
rule_arn = events.put_rule(Name=MORNING_RULE, ScheduleExpression=MORNING_CRON, State='ENABLED',
                           Description='글로벌 DB 봇 — 승인 대기 아침 요약 (08:00 KST)')['RuleArn']
events.put_targets(Rule=MORNING_RULE, Targets=[{'Id': 'bot', 'Arn': bot_arn, 'Input': json.dumps({'action': 'morning'})}])
try:
    lam.add_permission(FunctionName=BOT_FUNCTION, StatementId='events-morning', Action='lambda:InvokeFunction',
                       Principal='events.amazonaws.com', SourceArn=rule_arn)
    print('EventBridge 호출 권한 추가')
except lam.exceptions.ResourceConflictException:
    print('EventBridge 호출 권한 있음')
print('규칙:', MORNING_RULE, MORNING_CRON)

5. `dev` 스테이지 배포 · webhook 등록 · 확인

In [ ]:
apigw.create_deployment(restApiId=API_ID, stageName=STAGE, description='GDB bot: /tgbot, /regreview (Cognito)')

tg('setWebhook', url=f'{BASE_URL}/tgbot', secret_token=env['GDB_BOT_WEBHOOK_SECRET'],
   allowed_updates=['message', 'callback_query'], drop_pending_updates=True)
tg('setMyCommands', commands=[{'command': 'pending', 'description': '승인 대기 목록 (카드 다시 받기)'},
                              {'command': 'help', 'description': '안내'},
                              {'command': 'start', 'description': '시작 · 내 텔레그램 번호'}])
info = tg('getWebhookInfo')
print('webhook:', info['url'], '· 대기', info.get('pending_update_count'), '· 마지막 오류', info.get('last_error_message', '없음'))

def status_of(method, path, headers=None):
    # 새 경로는 1~2분 'Missing Authentication Token' 이 섞여 나온다 → 다시 시도
    for _ in range(15):
        req = urllib.request.Request(BASE_URL + path, method=method, headers=headers or {}, data=b'{}' if method == 'POST' else None)
        try:
            with urllib.request.urlopen(req) as r:
                return r.status, json.load(r)
        except urllib.error.HTTPError as e:
            body = e.read()
            if b'Missing Authentication Token' not in body:
                return e.code, body[:100]
            time.sleep(8)
    raise RuntimeError(f'{path}: 경로가 반영되지 않았습니다')

print('로그인 없이 /regreview:', status_of('GET', '/regreview')[0])                         # 401
print('secret 없이 /tgbot:', status_of('POST', '/tgbot', {'Content-Type': 'application/json'}))  # Lambda 403 (무시)
print(f"이제 텔레그램에서 @{me['username']} 에게 /start 를 보내 번호를 확인하세요.")

6. 봇 승인자 텔레그램 번호만 바꿀 때 — `.env`의 `GDB_OWNER_TG`를 고친 뒤 이 셀만 (Settings 셀 다음에)

In [ ]:
env = read_env()
boto3.client('lambda').get_waiter('function_updated_v2').wait(FunctionName=BOT_FUNCTION)
current = boto3.client('lambda').get_function_configuration(FunctionName=BOT_FUNCTION)['Environment']['Variables']
boto3.client('lambda').update_function_configuration(
    FunctionName=BOT_FUNCTION, Environment={'Variables': {**current, 'GDB_OWNER_TG': env.get('GDB_OWNER_TG', '')}})
print('봇 승인자:', len([x for x in env.get('GDB_OWNER_TG', '').split(',') if x.strip()]), '명 — 봇에 /start 를 다시 보내 확인')